# Syria Administrative Point Spatial Join Map

Overview

Assigns governorate attributes to administrative reference points in Syria using a point-in-polygon spatial join.
The national-level reference point is excluded because it does not belong to a single governorate, leaving 348 governorate, district and sub-district points for analysis.
A left spatial join with the `within` predicate preserves every analytical point and allows unmatched or multiple assignments to be detected.
The spatially assigned governorate attributes are compared with the governorate attributes already recorded in the source point dataset.

Point-in-PolygonによるSpatial Joinを用いて、シリアの行政地点に所属県の属性を付与します。
国レベルの代表地点は単一の県に所属しないため除外し、県、郡および準郡を表す348地点を分析対象とします。
`within`を空間述語とする左結合を使用し、すべての分析対象地点を保持したまま、未結合地点や複数の県に結合された地点を検出します。
Spatial Joinで取得した県属性は、元の地点データに記録されている県属性と照合します。

Objectives

- Read and validate the administrative point and boundary datasets
- Exclude the national-level reference point from the analysis
- Perform a left point-in-polygon spatial join using the `within` predicate
- Detect unmatched points and points assigned to multiple governorates
- Compare the spatially assigned governorate attributes with the source attributes
- Save the verified spatial join result as a derived GeoPackage
- Visualise the administrative points by administrative level
- Create an interactive Folium map with information panels, a legend and layer controls

- 行政地点データと行政界データを読み込み、検証する
- 国レベルの代表地点を分析対象から除外する
- `within`を空間述語としてPoint-in-Polygonによる左Spatial Joinを実行する
- 未結合地点および複数の県に結合された地点を検出する
- Spatial Joinで取得した県属性と元の県属性を照合する
- 検証済みのSpatial Join結果を派生GeoPackageとして保存する
- 行政レベル別に地点を可視化する
- 情報パネル、凡例およびレイヤー切り替え機能を備えたFolium地図を作成する

Workflow

1. Define the input and output file paths
2. Read the administrative point and boundary datasets
3. Validate the coordinate systems, required attributes and geometries
4. Exclude the national-level reference point
5. Prepare the governorate attributes for the spatial join
6. Perform a left spatial join using the `within` predicate
7. Detect unmatched and multiply assigned points
8. Compare the source and spatially assigned governorate attributes
9. Save and verify the derived GeoPackage
10. Create the administrative-level point layers
11. Add boundaries, labels, information panels, a legend and layer controls
12. Save and display the interactive map

1. 入出力ファイルのパスを設定する
2. 行政地点データと行政界データを読み込む
3. 座標参照系、必須属性およびジオメトリを検証する
4. 国レベルの代表地点を分析対象から除外する
5. Spatial Joinで使用する県属性を準備する
6. `within`を空間述語として左Spatial Joinを実行する
7. 未結合地点および複数の県に結合された地点を検出する
8. 元の県属性とSpatial Joinで取得した県属性を照合する
9. 派生GeoPackageを保存し、内容を確認する
10. 行政レベル別の地点レイヤーを作成する
11. 行政界、ラベル、情報パネル、凡例およびレイヤー切り替え機能を追加する
12. インタラクティブ地図を保存し、Notebook上に表示する

Data

Administrative point data:

- `syr_adminpoints.geojson`

Administrative boundary data:

- `syr_admin0.geojson`
- `syr_admin1.geojson`

Source: HDX OCHA, Syria subnational administrative boundaries

Technologies

- Python
- GeoPandas
- Folium
- GeoPackage

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

from pathlib import Path

import folium
import geopandas as gpd

In [ ]:
# 2
# Define the input and output file paths
# 入力データと出力ファイルのパスを設定する

PROJECT_DIR = Path.cwd()
ROOT_DIR = PROJECT_DIR.parents[1]

VECTOR_DIR = ROOT_DIR / "02_DATA" / "VECTOR"
OUTPUT_DIR = PROJECT_DIR / "outputs"

adminpoints_path = (
    VECTOR_DIR / "syr_adminpoints.geojson"
)

admin0_path = (
    VECTOR_DIR / "syr_admin0.geojson"
)

admin1_path = (
    VECTOR_DIR / "syr_admin1.geojson"
)

spatial_join_path = (
    OUTPUT_DIR / "syr_adminpoints_with_governorate.gpkg"
)

output_path = (
    PROJECT_DIR / "03_syria_vector_spatial_join.html"
)

print(f"Administrative point dataset: {adminpoints_path}")
print(f"Country boundary dataset: {admin0_path}")
print(f"Governorate boundary dataset: {admin1_path}")
print(f"Derived spatial join dataset: {spatial_join_path}")
print(f"Interactive map: {output_path}")

In [ ]:
# 3
# Read the administrative point and boundary datasets
# 行政地点データと行政界データを読み込む

adminpoints = gpd.read_file(
    adminpoints_path
)

admin0 = gpd.read_file(
    admin0_path
)

admin1 = gpd.read_file(
    admin1_path
)

print(
    "Administrative point features:",
    f"{len(adminpoints):,}"
)

print(
    "Country boundary features:",
    f"{len(admin0):,}"
)

print(
    "Governorate boundary features:",
    f"{len(admin1):,}"
)

In [ ]:
# 4
# Validate the coordinate reference systems
# 各データの座標参照系を検証する

spatial_datasets = {
    "Administrative points": adminpoints,
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

for dataset_name, dataset in spatial_datasets.items():

    if dataset.crs is None:
        raise ValueError(
            f"{dataset_name} has no defined CRS."
        )

    print(
        f"{dataset_name} CRS: {dataset.crs}"
    )

if not (
    adminpoints.crs
    == admin0.crs
    == admin1.crs
):
    raise ValueError(
        "The administrative point and boundary "
        "datasets must use the same CRS."
    )

if adminpoints.crs.to_epsg() != 4326:
    raise ValueError(
        "The spatial datasets were expected to use "
        f"EPSG:4326, but their CRS is {adminpoints.crs}."
    )

In [ ]:
# 5
# Validate the administrative point dataset
# 行政地点データの属性とジオメトリを検証する

required_point_columns = {
    "admin_level",
    "name",
    "x_coord",
    "y_coord",
    "adm1_name",
    "adm1_pcode",
    "geometry",
}

missing_point_columns = (
    required_point_columns
    - set(adminpoints.columns)
)

if missing_point_columns:
    raise ValueError(
        "The administrative point dataset is missing "
        f"required columns: {sorted(missing_point_columns)}"
    )

if len(adminpoints) != 349:
    raise ValueError(
        "Exactly 349 administrative point features "
        f"were expected, but {len(adminpoints)} were found."
    )

if adminpoints[
    [
        "x_coord",
        "y_coord",
    ]
].isna().any().any():
    raise ValueError(
        "The administrative point dataset contains "
        "missing coordinate attributes."
    )

if adminpoints.geometry.isna().any():
    raise ValueError(
        "The administrative point dataset contains "
        "missing geometries."
    )

if adminpoints.geometry.is_empty.any():
    raise ValueError(
        "The administrative point dataset contains "
        "empty geometries."
    )

if not adminpoints.geometry.is_valid.all():

    invalid_point_count = int(
        (
            ~adminpoints.geometry.is_valid
        ).sum()
    )

    raise ValueError(
        "The administrative point dataset contains "
        f"{invalid_point_count:,} invalid geometries."
    )

unexpected_point_types = (
    set(adminpoints.geom_type.unique())
    - {"Point"}
)

if unexpected_point_types:
    raise ValueError(
        "Unexpected administrative point geometry "
        f"types were found: {sorted(unexpected_point_types)}"
    )

# Measure the differences between the stored coordinates
# and the point geometry coordinates.
# 保存された座標属性とPointジオメトリの座標差を計測する

longitude_difference = (
    adminpoints.geometry.x
    - adminpoints["x_coord"]
).abs()

latitude_difference = (
    adminpoints.geometry.y
    - adminpoints["y_coord"]
).abs()

maximum_longitude_difference = float(
    longitude_difference.max()
)

maximum_latitude_difference = float(
    latitude_difference.max()
)

# Allow small differences caused by coordinate rounding.
# 座標の丸めによって生じる小さな差を許容する

COORDINATE_TOLERANCE_DEGREES = 1e-8

if (
    longitude_difference
    > COORDINATE_TOLERANCE_DEGREES
).any():
    raise ValueError(
        "One or more point longitude attributes differ "
        "from their geometry coordinates beyond the "
        "allowed tolerance."
    )

if (
    latitude_difference
    > COORDINATE_TOLERANCE_DEGREES
).any():
    raise ValueError(
        "One or more point latitude attributes differ "
        "from their geometry coordinates beyond the "
        "allowed tolerance."
    )

print(
    "Maximum longitude difference:",
    f"{maximum_longitude_difference:.12f}°"
)

print(
    "Maximum latitude difference:",
    f"{maximum_latitude_difference:.12f}°"
)

print(
    "Coordinate tolerance:",
    f"{COORDINATE_TOLERANCE_DEGREES:.12f}°"
)

print(
    adminpoints.geom_type.value_counts()
)

print(
    adminpoints[
        "admin_level"
    ].value_counts().sort_index()
)

In [ ]:
# 6
# Validate the administrative boundary datasets
# 国境データと県境データを検証する

required_admin0_columns = {
    "adm0_name",
    "adm0_pcode",
    "geometry",
}

required_admin1_columns = {
    "adm1_name",
    "adm1_pcode",
    "center_lat",
    "center_lon",
    "geometry",
}

missing_admin0_columns = (
    required_admin0_columns
    - set(admin0.columns)
)

missing_admin1_columns = (
    required_admin1_columns
    - set(admin1.columns)
)

if missing_admin0_columns:
    raise ValueError(
        "The country boundary dataset is missing "
        f"required columns: {sorted(missing_admin0_columns)}"
    )

if missing_admin1_columns:
    raise ValueError(
        "The governorate boundary dataset is missing "
        f"required columns: {sorted(missing_admin1_columns)}"
    )

if len(admin0) != 1:
    raise ValueError(
        "Exactly one country boundary feature was expected, "
        f"but {len(admin0)} features were found."
    )

if len(admin1) != 14:
    raise ValueError(
        "Exactly 14 governorate boundary features were expected, "
        f"but {len(admin1)} features were found."
    )

for dataset_name, dataset in {
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}.items():

    if dataset.geometry.isna().any():
        raise ValueError(
            f"{dataset_name} contains missing geometries."
        )

    if dataset.geometry.is_empty.any():
        raise ValueError(
            f"{dataset_name} contains empty geometries."
        )

    if not dataset.geometry.is_valid.all():

        invalid_count = int(
            (
                ~dataset.geometry.is_valid
            ).sum()
        )

        raise ValueError(
            f"{dataset_name} contains "
            f"{invalid_count:,} invalid geometries."
        )

    unexpected_geometry_types = (
        set(dataset.geom_type.unique())
        - {
            "Polygon",
            "MultiPolygon",
        }
    )

    if unexpected_geometry_types:
        raise ValueError(
            f"{dataset_name} contains unexpected "
            "geometry types: "
            f"{sorted(unexpected_geometry_types)}"
        )

if admin1[
    [
        "center_lat",
        "center_lon",
    ]
].isna().any().any():
    raise ValueError(
        "The governorate label coordinates "
        "contain missing values."
    )

print(
    admin1[
        [
            "adm1_name",
            "adm1_pcode",
        ]
    ].sort_values(
        "adm1_name"
    )
)

In [ ]:
# 7
# Exclude the national-level reference point
# 国レベルの代表地点を除外する

ANALYTICAL_ADMIN_LEVELS = {
    1,
    2,
    3,
}

EXPECTED_LEVEL_COUNTS = {
    1: 14,
    2: 62,
    3: 272,
}

adminpoints_analysis = (
    adminpoints[
        adminpoints[
            "admin_level"
        ].isin(
            ANALYTICAL_ADMIN_LEVELS
        )
    ].copy()
)

excluded_adminpoints = (
    adminpoints[
        ~adminpoints[
            "admin_level"
        ].isin(
            ANALYTICAL_ADMIN_LEVELS
        )
    ].copy()
)

actual_level_counts = (
    adminpoints_analysis[
        "admin_level"
    ].value_counts().sort_index().to_dict()
)

if actual_level_counts != EXPECTED_LEVEL_COUNTS:
    raise ValueError(
        "The administrative-level point counts do not "
        "match the expected composition. "
        f"Actual: {actual_level_counts}; "
        f"Expected: {EXPECTED_LEVEL_COUNTS}"
    )

if len(excluded_adminpoints) != 1:
    raise ValueError(
        "Exactly one national-level reference point "
        f"was expected to be excluded, but "
        f"{len(excluded_adminpoints)} points were excluded."
    )

if set(
    excluded_adminpoints[
        "admin_level"
    ]
) != {0}:
    raise ValueError(
        "The excluded point is not the expected "
        "national-level reference point."
    )

if adminpoints_analysis[
    [
        "adm1_name",
        "adm1_pcode",
    ]
].isna().any().any():
    raise ValueError(
        "One or more analytical points are missing "
        "source governorate attributes."
    )

print(
    "Analytical point features:",
    f"{len(adminpoints_analysis):,}"
)

print(
    "Excluded national-level features:",
    f"{len(excluded_adminpoints):,}"
)

print(
    "Analytical point composition:"
)

print(
    adminpoints_analysis[
        "admin_level"
    ].value_counts().sort_index()
)

print(
    "Excluded point:"
)

print(
    excluded_adminpoints[
        [
            "admin_level",
            "name",
        ]
    ]
)

In [ ]:
# 8
# Prepare the governorate attributes for the spatial join
# Spatial Joinで使用する県属性を準備する

admin1_join = (
    admin1[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ].rename(
        columns={
            "adm1_name": "spatial_adm1_name",
            "adm1_pcode": "spatial_adm1_pcode",
        }
    ).copy()
)

print(
    admin1_join[
        [
            "spatial_adm1_name",
            "spatial_adm1_pcode",
        ]
    ].sort_values(
        "spatial_adm1_name"
    )
)

In [ ]:
# 9
# Perform a left point-in-polygon spatial join
# withinを空間述語として左Spatial Joinを実行する

joined_adminpoints = gpd.sjoin(
    adminpoints_analysis,
    admin1_join,
    how="left",
    predicate="within",
)

print(
    "Spatial join result rows:",
    f"{len(joined_adminpoints):,}"
)

print(
    joined_adminpoints[
        [
            "admin_level",
            "name",
            "adm1_name",
            "adm1_pcode",
            "spatial_adm1_name",
            "spatial_adm1_pcode",
        ]
    ].head()
)

In [ ]:
# 10
# Detect administrative points without a governorate assignment
# 県へ結合されなかった行政地点を検出する

unmatched_point_mask = (
    joined_adminpoints[
        "spatial_adm1_pcode"
    ].isna()
)

unmatched_point_count = int(
    unmatched_point_mask.sum()
)

if unmatched_point_count > 0:

    print(
        joined_adminpoints.loc[
            unmatched_point_mask,
            [
                "admin_level",
                "name",
                "adm1_name",
                "adm1_pcode",
            ]
        ]
    )

    raise ValueError(
        "The spatial join left "
        f"{unmatched_point_count:,} administrative "
        "points without a governorate assignment."
    )

print(
    "Unmatched administrative points:",
    f"{unmatched_point_count:,}"
)

In [ ]:
# 11
# Detect multiple assignments and validate the result count
# 複数の県への結合とSpatial Join後の件数を検証する

join_count_by_source_point = (
    joined_adminpoints.groupby(
        level=0
    ).size()
)

multiply_matched_point_indices = (
    join_count_by_source_point[
        join_count_by_source_point > 1
    ].index
)

multiply_matched_point_count = len(
    multiply_matched_point_indices
)

if multiply_matched_point_count > 0:

    print(
        joined_adminpoints.loc[
            multiply_matched_point_indices,
            [
                "admin_level",
                "name",
                "spatial_adm1_name",
                "spatial_adm1_pcode",
            ]
        ]
    )

    raise ValueError(
        f"{multiply_matched_point_count:,} administrative "
        "points were assigned to multiple governorates."
    )

if len(joined_adminpoints) != len(
    adminpoints_analysis
):
    raise ValueError(
        "The spatial join output row count does not "
        "match the analytical input point count."
    )

print(
    "Input analytical points:",
    f"{len(adminpoints_analysis):,}"
)

print(
    "Spatial join result rows:",
    f"{len(joined_adminpoints):,}"
)

print(
    "Multiply matched points:",
    f"{multiply_matched_point_count:,}"
)

In [ ]:
# 12
# Compare the source and spatially assigned governorate attributes
# 元の県属性とSpatial Joinで取得した県属性を照合する

governorate_name_match = (
    joined_adminpoints[
        "adm1_name"
    ]
    == joined_adminpoints[
        "spatial_adm1_name"
    ]
)

governorate_pcode_match = (
    joined_adminpoints[
        "adm1_pcode"
    ]
    == joined_adminpoints[
        "spatial_adm1_pcode"
    ]
)

joined_adminpoints[
    "assignment_match"
] = (
    governorate_name_match
    & governorate_pcode_match
)

assignment_mismatch_mask = (
    ~joined_adminpoints[
        "assignment_match"
    ]
)

assignment_mismatch_count = int(
    assignment_mismatch_mask.sum()
)

if assignment_mismatch_count > 0:

    print(
        joined_adminpoints.loc[
            assignment_mismatch_mask,
            [
                "admin_level",
                "name",
                "adm1_name",
                "adm1_pcode",
                "spatial_adm1_name",
                "spatial_adm1_pcode",
            ]
        ]
    )

    raise ValueError(
        "The source and spatially assigned governorate "
        "attributes disagree for "
        f"{assignment_mismatch_count:,} points."
    )

print(
    "Matching governorate assignments:",
    f"{int(joined_adminpoints['assignment_match'].sum()):,}"
)

print(
    "Mismatched governorate assignments:",
    f"{assignment_mismatch_count:,}"
)

In [ ]:
# 13
# Prepare and save the verified spatial join result
# 検証済みのSpatial Join結果を整理して保存する

ADMIN_LEVEL_LABELS = {
    1: "Governorate",
    2: "District",
    3: "Sub-District",
}

joined_verified = (
    joined_adminpoints.drop(
        columns=[
            "index_right",
        ]
    ).copy()
)

joined_verified[
    "admin_level_name"
] = (
    joined_verified[
        "admin_level"
    ].map(
        ADMIN_LEVEL_LABELS
    )
)

if joined_verified[
    "admin_level_name"
].isna().any():
    raise ValueError(
        "One or more administrative points have "
        "an unknown administrative level."
    )

# Preserve the source point index for traceability.
# 元データの地点を確認できるようにインデックスを保持する

joined_verified.insert(
    0,
    "source_point_index",
    joined_verified.index.astype(int),
)

joined_verified = (
    joined_verified.reset_index(
        drop=True
    )
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

joined_verified.to_file(
    spatial_join_path,
    layer="adminpoints_with_governorate",
    driver="GPKG",
    index=False,
)

print(
    "Verified spatial join dataset saved to:",
    spatial_join_path,
)

In [ ]:
# 14
# Read and verify the saved spatial join dataset
# 保存したSpatial Joinデータを読み込み、内容を確認する

saved_joined_adminpoints = gpd.read_file(
    spatial_join_path,
    layer="adminpoints_with_governorate",
)

required_saved_columns = {
    "source_point_index",
    "admin_level",
    "admin_level_name",
    "name",
    "adm1_name",
    "adm1_pcode",
    "spatial_adm1_name",
    "spatial_adm1_pcode",
    "assignment_match",
    "geometry",
}

missing_saved_columns = (
    required_saved_columns
    - set(saved_joined_adminpoints.columns)
)

if missing_saved_columns:
    raise ValueError(
        "The saved spatial join dataset is missing "
        f"required columns: {sorted(missing_saved_columns)}"
    )

if len(saved_joined_adminpoints) != len(
    adminpoints_analysis
):
    raise ValueError(
        "The saved feature count does not match "
        "the verified spatial join result."
    )

if (
    saved_joined_adminpoints.crs
    != adminpoints_analysis.crs
):
    raise ValueError(
        "The saved spatial join CRS does not match "
        "the source point CRS."
    )

if saved_joined_adminpoints[
    [
        "spatial_adm1_name",
        "spatial_adm1_pcode",
    ]
].isna().any().any():
    raise ValueError(
        "The saved spatial join dataset contains "
        "missing governorate assignments."
    )

if not saved_joined_adminpoints[
    "assignment_match"
].all():
    raise ValueError(
        "The saved spatial join dataset contains "
        "a mismatched governorate assignment."
    )

unusable_geometry_mask = (
    saved_joined_adminpoints.geometry.isna()
    | saved_joined_adminpoints.geometry.is_empty
    | ~saved_joined_adminpoints.geometry.is_valid
)

if unusable_geometry_mask.any():
    raise ValueError(
        "The saved spatial join dataset contains "
        "missing, empty or invalid geometries."
    )

print(
    "Saved spatial join features:",
    f"{len(saved_joined_adminpoints):,}"
)

print(
    "Saved spatial join CRS:",
    saved_joined_adminpoints.crs,
)

print(
    "Verified assignment matches:",
    f"{int(saved_joined_adminpoints['assignment_match'].sum()):,}"
)

print(
    "Saved administrative-level composition:"
)

print(
    saved_joined_adminpoints[
        "admin_level_name"
    ].value_counts()
)

In [ ]:
# 15
# Define the point styles and create the map
# 行政レベル別の表示設定と地図を作成する

ADMIN_LEVEL_STYLES = {
    1: {
        "label": "Governorate",
        "color": "#720f32",
        "radius": 6,
    },
    2: {
        "label": "District",
        "color": "#936f27",
        "radius": 4,
    },
    3: {
        "label": "Sub-District",
        "color": "#114665",
        "radius": 2.5,
    },
}

m = folium.Map(
    location=[
        34.8,
        38.5,
    ],
    zoom_start=7,
    tiles=None,
)

folium.TileLayer(
    tiles=(
        "https://{s}.basemaps.cartocdn.com/"
        "light_nolabels/{z}/{x}/{y}{r}.png"
    ),
    attr=(
        '&copy; <a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        '&copy; <a href="https://carto.com/attributions">'
        "CARTO</a>"
    ),
    name="CARTO Light — No Labels",
    overlay=False,
    control=True,
).add_to(
    m
)

min_x, min_y, max_x, max_y = (
    admin0.total_bounds
)

syria_view_bounds = [
    [
        min_y,
        min_x,
    ],
    [
        max_y,
        max_x,
    ],
]

m.fit_bounds(
    syria_view_bounds,
    padding=(35, 35),
    max_zoom=7,
)

In [ ]:
# 16
# Add the country and governorate boundaries
# 国境および県境を追加する

folium.GeoJson(
    admin1[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ],
    name="Governorate Boundaries",
    style_function=lambda feature: {
        "color": "#777777",
        "weight": 1,
        "fillOpacity": 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(
    m
)

folium.GeoJson(
    admin0[
        [
            "adm0_name",
            "adm0_pcode",
            "geometry",
        ]
    ],
    name="Syria Boundary",
    style_function=lambda feature: {
        "color": "#222222",
        "weight": 2.5,
        "fillOpacity": 0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm0_name",
            "adm0_pcode",
        ],
        aliases=[
            "Country:",
            "Pcode:",
        ],
        sticky=False,
        labels=True,
    ),
).add_to(
    m
)

In [ ]:
# 17
# Add the administrative points by administrative level
# 行政レベル別に地点レイヤーを追加する

point_feature_groups = {}

for admin_level in [
    3,
    2,
    1,
]:

    level_style = (
        ADMIN_LEVEL_STYLES[
            admin_level
        ]
    )

    level_points = (
        saved_joined_adminpoints[
            saved_joined_adminpoints[
                "admin_level"
            ] == admin_level
        ]
    )

    point_layer = folium.FeatureGroup(
        name=(
            f"{level_style['label']} Points "
            f"({len(level_points):,})"
        ),
        show=True,
    )

    for _, point in level_points.iterrows():

        popup_html = f"""
        <div style="
            min-width: 210px;
            font-size: 12px;
            line-height: 1.4;
        ">
            <b>{point["name"]}</b><br>
            Administrative level:
            {point["admin_level_name"]}<br>
            Spatially assigned governorate:
            {point["spatial_adm1_name"]}<br>
            Governorate Pcode:
            {point["spatial_adm1_pcode"]}<br>
            Source assignment verified:
            {point["assignment_match"]}
        </div>
        """

        folium.CircleMarker(
            location=[
                point.geometry.y,
                point.geometry.x,
            ],
            radius=level_style[
                "radius"
            ],
            color=level_style[
                "color"
            ],
            weight=1,
            fill=True,
            fill_color=level_style[
                "color"
            ],
            fill_opacity=0.82,
            tooltip=folium.Tooltip(
                (
                    f"{point['name']} — "
                    f"{point['admin_level_name']}"
                ),
                sticky=False,
            ),
            popup=folium.Popup(
                popup_html,
                max_width=300,
            ),
        ).add_to(
            point_layer
        )

    point_layer.add_to(
        m
    )

    point_feature_groups[
        admin_level
    ] = point_layer

In [ ]:
# 18
# Add neighbouring country labels
# 周辺国名を追加する

neighbour_label_layer = folium.FeatureGroup(
    name="Neighbour Labels",
    show=True,
)

neighbour_labels = {
    "TÜRKIYE": [37.5, 37.5],
    "IRAQ": [34.5, 42.0],
    "JORDAN": [31.9, 36.5],
    "LEBANON": [34.2, 35.0],
}

for country_name, coordinates in (
    neighbour_labels.items()
):

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            html=f"""
            <div style="
                width: 120px;
                margin-left: -60px;
                color: #666666;
                font-size: 14pt;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                text-shadow:
                    -1px -1px 0 white,
                    1px -1px 0 white,
                    -1px 1px 0 white,
                    1px 1px 0 white;
            ">
                {country_name}
            </div>
            """
        ),
    ).add_to(
        neighbour_label_layer
    )

neighbour_label_layer.add_to(
    m
)

In [ ]:
# 19
# Add governorate labels
# 県名を追加する

governorate_label_layer = folium.FeatureGroup(
    name="Governorate Labels",
    show=True,
)

for _, governorate in admin1.iterrows():

    folium.Marker(
        location=[
            governorate["center_lat"],
            governorate["center_lon"],
        ],
        icon=folium.DivIcon(
            html=f"""
            <div style="
                width: 120px;
                margin-left: -60px;
                color: #222222;
                font-size: 10pt;
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                text-shadow:
                    -1px -1px 0 white,
                    1px -1px 0 white,
                    -1px 1px 0 white,
                    1px 1px 0 white;
            ">
                {governorate["adm1_name"]}
            </div>
            """
        ),
    ).add_to(
        governorate_label_layer
    )

governorate_label_layer.add_to(
    m
)

In [ ]:
# 20
# Add the map information and validation panel
# 地図の説明、Spatial Join方法および検証結果を追加する

verified_assignment_count = int(
    saved_joined_adminpoints[
        "assignment_match"
    ].sum()
)

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 430px;
    min-height: 230px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="
        color: #0077b6;
        font-weight: bold;
    ">
        Administrative Point Spatial Join
    </span>

    <small style="
        display: block;
        margin-top: 7px;
        line-height: 1.35;
        color: #333333;
    ">
        Governorate attributes were assigned to
        governorate, district and sub-district
        reference points using a left point-in-polygon
        spatial join with the within predicate.
        The national-level reference point was excluded
        because it does not belong to a single governorate.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        Point and boundary source:
        <b>HDX OCHA</b><br>

        Source point features:
        <b>{len(adminpoints):,}</b><br>

        Analytical point features:
        <b>{len(saved_joined_adminpoints):,}</b><br>

        Unmatched points:
        <b>{unmatched_point_count:,}</b><br>

        Multiply matched points:
        <b>{multiply_matched_point_count:,}</b><br>

        Verified source/spatial assignments:
        <b>{verified_assignment_count:,}</b><br>

        Method:
        Left Spatial Join / within / Attribute Verification
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        information_panel_html
    )
)

In [ ]:
# 21
# Add the administrative-level point legend
# 行政レベル別の地点凡例を追加する

legend_items_html = ""

for admin_level in [
    1,
    2,
    3,
]:

    level_style = (
        ADMIN_LEVEL_STYLES[
            admin_level
        ]
    )

    level_count = (
        EXPECTED_LEVEL_COUNTS[
            admin_level
        ]
    )

    marker_diameter = (
        level_style[
            "radius"
        ] * 2
    )

    legend_items_html += f"""
    <div style="
        display: flex;
        align-items: center;
        margin-top: 9px;
    ">
        <span style="
            display: inline-block;
            width: {marker_diameter}px;
            height: {marker_diameter}px;
            margin-left: {12 - level_style["radius"]}px;
            margin-right: {12 - level_style["radius"] + 9}px;
            background-color: {level_style["color"]};
            border: 1px solid {level_style["color"]};
            border-radius: 50%;
            opacity: 0.85;
        "></span>

        {level_style["label"]} — {level_count:,}
    </div>
    """

legend_html = f"""
<div style="
    position: fixed;
    bottom: 40px;
    right: 40px;
    width: 255px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    font-size: 12px;
    z-index: 9999;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.2);
">
    <b style="font-size: 13px;">
        Administrative Reference Points
    </b>

    {legend_items_html}

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        color: #555555;
        border-top: 1px solid #aaaaaa;
        line-height: 1.35;
    ">
        Total analytical points:
        {len(saved_joined_adminpoints):,}<br>

        National-level points excluded:
        {len(excluded_adminpoints):,}
    </div>
</div>
"""

m.get_root().html.add_child(
    folium.Element(
        legend_html
    )
)

In [ ]:
# 22
# Add the layer control
# レイヤー切り替え機能を追加する

folium.LayerControl(
    collapsed=False,
).add_to(
    m
)

In [ ]:
# 23
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(
    output_path
)

print(
    f"Map saved to: {output_path}"
)

m